# Salary Prediction with Linear Regression

**Author:** Patricio Gonzalez Ramos  
**Date:** February 2026  
**Dataset:** `Salary_dataset.csv`

---

## Overview

This project builds a simple linear regression model to predict employee salary based on years of experience. The pipeline covers end-to-end steps: exploratory data analysis, preprocessing with Min-Max normalization, model training, and evaluation using standard regression metrics.

**Key question:** *How well can years of experience alone explain salary variation?*

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

## 2. Load & Explore the Data

The dataset contains two columns: `YearsExperience` and `Salary`. We start with a quick look at structure, summary statistics, and data quality.

In [ ]:
df = pd.read_csv('data/Salary_dataset.csv', index_col=0)

print("First 5 rows:")
print(df.head())

print("\nDataset info:")
print(df.info())

print("\nStatistical summary:")
print(df.describe())

print("\nMissing values:")
print(df.isnull().sum())

## 3. Data Cleaning

Before modeling, we check for missing values and duplicate rows. Clean data is essential for reliable regression results.

In [ ]:
# Handle missing values
if df.isnull().sum().sum() == 0:
    print("✓ No missing values found")
else:
    print("Missing values found — removing them")
    df = df.dropna()

# Handle duplicates
duplicates = df.duplicated().sum()
if duplicates == 0:
    print("✓ No duplicate rows found")
else:
    print(f"Found {duplicates} duplicates — removing them")
    df = df.drop_duplicates()

## 4. Correlation Analysis

We compute the Pearson correlation matrix to quantify the linear relationship between experience and salary. A correlation close to 1 would indicate a strong positive linear relationship — making linear regression a well-suited model for this problem.

In [ ]:
correlation_matrix = df.corr()
print("Correlation Matrix:")
print(correlation_matrix)

correlation = correlation_matrix.loc['YearsExperience', 'Salary']
print(f"\nCorrelation between Experience and Salary: {correlation:.4f}")

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='Blues',
            square=True, fmt='.3f', linewidths=2)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Preprocessing — Min-Max Normalization

Both features are scaled to the [0, 1] range using Min-Max normalization. This ensures that the model is not sensitive to the raw scale of salary values (which are in the tens of thousands) relative to years of experience (single digits). We keep separate scalers for `X` and `y` so we can inverse-transform predictions back to dollar values for interpretation.

In [ ]:
X = df[['YearsExperience']].values
y = df['Salary'].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_normalized = scaler_X.fit_transform(X)
y_normalized = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

print("After normalization:")
print(f"  Experience — Min: {X_normalized.min():.2f}, Max: {X_normalized.max():.2f}")
print(f"  Salary     — Min: {y_normalized.min():.2f}, Max: {y_normalized.max():.2f}")

## 6. Train / Test Split

The dataset is split 80/20 into training and testing sets. `random_state=42` ensures reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y_normalized, test_size=0.2, random_state=42
)

print(f"Training set : {len(X_train)} samples (80%)")
print(f"Testing set  : {len(X_test)} samples (20%)")

## 7. Model Training — Simple Linear Regression

We fit a simple linear regression model on the normalized training data using scikit-learn's `LinearRegression`. The resulting equation maps normalized experience to normalized salary.

In [ ]:
model = LinearRegression()
model.fit(X_train.reshape(-1, 1), y_train)

print(f"Model equation (normalized space):")
print(f"  ŷ = {model.coef_[0]:.4f}x + {model.intercept_:.4f}")

## 8. Predictions

Predictions are generated for both training and test sets. We inverse-transform everything back to original units (years and dollars) for human-readable sample output.

In [ ]:
y_train_pred = model.predict(X_train.reshape(-1, 1))
y_test_pred  = model.predict(X_test.reshape(-1, 1))

# Inverse-transform for interpretability
X_test_original      = scaler_X.inverse_transform(X_test.reshape(-1, 1))
y_test_original      = scaler_y.inverse_transform(y_test.reshape(-1, 1))
y_test_pred_original = scaler_y.inverse_transform(y_test_pred.reshape(-1, 1))

X_train_original      = scaler_X.inverse_transform(X_train.reshape(-1, 1))
y_train_original      = scaler_y.inverse_transform(y_train.reshape(-1, 1))
y_train_pred_original = scaler_y.inverse_transform(y_train_pred.reshape(-1, 1))

print("Sample predictions (first 3 test samples):")
for i in range(3):
    print(f"\n  Sample {i+1}:")
    print(f"    Experience      : {X_test_original[i][0]:.1f} years")
    print(f"    Actual Salary   : ${y_test_original[i][0]:,.0f}")
    print(f"    Predicted Salary: ${y_test_pred_original[i][0]:,.0f}")

## 9. Model Evaluation

We evaluate the model using three metrics:

| Metric | Description |
|---|---|
| **R² (Coefficient of Determination)** | Proportion of variance in salary explained by experience. Closer to 1 is better. |
| **MAE (Mean Absolute Error)** | Average absolute difference between predicted and actual salary. |
| **RMSE (Root Mean Squared Error)** | Like MAE, but penalizes large errors more heavily. |

We report both training and test R² to check for overfitting.

In [ ]:
train_r2  = r2_score(y_train, y_train_pred)
test_r2   = r2_score(y_test, y_test_pred)
test_mae  = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# Convert MAE and RMSE back to dollar scale
salary_range = y.max() - y.min()
test_mae_original  = test_mae  * salary_range
test_rmse_original = test_rmse * salary_range

print(f"Training R²  : {train_r2:.4f} ({train_r2*100:.2f}%)")
print(f"Testing R²   : {test_r2:.4f} ({test_r2*100:.2f}%)")
print(f"Test MAE     : ${test_mae_original:,.0f}")
print(f"Test RMSE    : ${test_rmse_original:,.0f}")

## 10. Visualizations

Three plots are generated to visually assess model fit and prediction quality.

In [ ]:
X_original = scaler_X.inverse_transform(X_normalized.reshape(-1, 1))
y_original = scaler_y.inverse_transform(y_normalized.reshape(-1, 1))

# Regression line in original space
X_line      = np.linspace(X_original.min(), X_original.max(), 100).reshape(-1, 1)
X_line_norm = scaler_X.transform(X_line)
y_line_pred = model.predict(X_line_norm)
y_line_orig = scaler_y.inverse_transform(y_line_pred.reshape(-1, 1))

# --- Plot 1: Scatter + Regression Line ---
plt.figure(figsize=(10, 6))
plt.scatter(X_original, y_original, color='steelblue', alpha=0.6, s=80, label='Actual Data')
plt.plot(X_line, y_line_orig, color='crimson', linewidth=2, label='Regression Line')
plt.xlabel('Years of Experience', fontsize=12)
plt.ylabel('Salary ($)', fontsize=12)
plt.title('Salary vs. Years of Experience — Linear Regression Fit', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('visuals/regression_fit.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Plot 2: Train/Test Split ---
plt.figure(figsize=(10, 6))
plt.scatter(X_train_original, y_train_original, color='steelblue', alpha=0.6, s=80, label='Training Data')
plt.scatter(X_test_original,  y_test_original,  color='seagreen',  alpha=0.6, s=80, label='Testing Data')
plt.plot(X_line, y_line_orig, color='crimson', linewidth=2, label='Regression Line')
plt.xlabel('Years of Experience', fontsize=12)
plt.ylabel('Salary ($)', fontsize=12)
plt.title('Training vs. Testing Data Split', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('visuals/train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Plot 3: Actual vs Predicted ---
plt.figure(figsize=(10, 6))
plt.scatter(y_test_original, y_test_pred_original, color='mediumpurple', alpha=0.6, s=80, label='Predictions')
min_val = min(y_test_original.min(), y_test_pred_original.min())
max_val = max(y_test_original.max(), y_test_pred_original.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Salary ($)', fontsize=12)
plt.ylabel('Predicted Salary ($)', fontsize=12)
plt.title('Actual vs. Predicted Salaries', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('visuals/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary & Key Takeaways

- **Strong linear relationship** — the high Pearson correlation confirms that years of experience is a reliable predictor of salary in this dataset.
- **High R² on both sets** — training and test R² are close, indicating the model generalizes well and is not overfit.
- **Min-Max scaling** — normalizing both features before fitting ensures the model is numerically stable, and inverse-transforming predictions restores full interpretability in dollar terms.
- **Limitation** — the model uses a single feature. Real-world salary prediction would benefit from additional variables (education level, industry, role, location) and possibly non-linear models.